# ArrhythmiaGuard — Complete Pipeline
### ESP32-S3 N16R8 | AAMI EC57 | MIT-BIH | RF + 1D CNN → TFLite

**Run order:** Cell 1 → Restart Runtime → Cell 2 onwards (top to bottom)

| Step | Cell | Description |
|------|------|-------------|
| 1 | Cell 1 | Install dependencies |
| 2 | Cell 2 | Imports + constants |
| 3 | Cell 3 | Download MIT-BIH records |
| 4 | Cell 4 | Preprocessing pipeline |
| 5 | Cell 5 | Feature extraction (17 features) |
| 6 | Cell 6 | Build DS1 + DS2 datasets |
| 7 | Cell 7 | SMOTE + Scale |
| 8 | Cell 8 | Random Forest — patient CV + train |
| 9 | Cell 9 | RF evaluate DS2 + threshold |
|10 | Cell 10 | SHAP explainability |
|11 | Cell 11 | Extract raw beats for CNN |
|12 | Cell 12 | Build + Train 1D CNN |
|13 | Cell 13 | Evaluate CNN on DS2 |
|14 | Cell 14 | Full visualisation |
|15 | Cell 15 | TFLite INT8 → C header |
|16 | Cell 16 | Save all models |
|17 | Cell 17 | Download everything |


## Cell 1 — Install
> ⚠️ After this cell finishes: **Runtime → Restart session** then run Cell 2 onwards

In [ ]:
!pip install wfdb numpy scipy scikit-learn \
             imbalanced-learn shap matplotlib \
             tqdm joblib tensorflow -q
print("DONE — Go to Runtime → Restart session")
print("Then run Cell 2 onwards. Do NOT re-run Cell 1.")

## Cell 2 — Imports & Constants

In [ ]:
import os, csv, json, warnings, joblib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import wfdb
from scipy.signal  import butter, filtfilt, iirnotch
from scipy.stats   import skew, kurtosis, entropy as sp_entropy
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing   import StandardScaler
from sklearn.impute          import SimpleImputer
from sklearn.metrics         import (
    confusion_matrix, roc_auc_score, roc_curve,
    f1_score, classification_report)
from imblearn.over_sampling import SMOTE
from tqdm import tqdm
import shap
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

FS     = 360
BEFORE = int(0.28 * FS)   # 100 samples before R-peak
AFTER  = int(0.44 * FS)   # 158 samples after R-peak
BLEN   = BEFORE + AFTER   # 258 samples total

print(f"TensorFlow  : {tf.__version__}")
print(f"Beat window : {BLEN} samples = {BLEN/FS*1000:.1f}ms")
print(f"Sample rate : {FS}Hz")
print("Cell 2 done")

## Cell 3 — AAMI EC57 Records + Download
> De Chazal 2004 inter-patient protocol. Excludes paced records: 102, 104, 107, 217

In [ ]:
DS1 = ['101','106','108','109','112','114','115',
       '116','118','119','122','124','201','203',
       '205','207','208','209','215','220','223']

DS2 = ['100','103','105','111','113','117','121',
       '123','200','202','210','212','213','214',
       '219','221','222','228','230','231']

PACED = ['102','104','107','217']
DS1   = [r for r in DS1 if r not in PACED]
DS2   = [r for r in DS2 if r not in PACED]

AAMI_MAP = {
    'N':'N','L':'N','R':'N','e':'N','j':'N',
    'A':'S','a':'S','J':'S','S':'S',
    'V':'V','E':'V',
    'F':'F',
    '/':'Q','f':'Q','Q':'Q','?':'Q'
}
LABEL_MAP = {'N':0,'S':1,'V':1,'F':1}

os.makedirs('mitbih', exist_ok=True)
ALL_RECORDS = list(set(DS1 + DS2))
print(f"Downloading {len(ALL_RECORDS)} records...")
print(f"DS1 Training : {len(DS1)} patients")
print(f"DS2 Testing  : {len(DS2)} patients")
print(f"Excluded paced: {PACED}\n")

for rec in tqdm(ALL_RECORDS, desc='Downloading'):
    try:
        wfdb.dl_database('mitdb', dl_dir='mitbih', records=[rec])
    except Exception as e:
        print(f"  {rec}: {e}")

print("Download complete")

## Cell 4 — Preprocessing Pipeline
> Bandpass 0.5–40Hz + 50Hz Notch (India) + Max Normalize

In [ ]:
def preprocess(sig, fs=360):
    nyq = fs / 2.0
    b, a = butter(4, [0.5/nyq, 40.0/nyq], btype='band')
    sig  = filtfilt(b, a, sig)
    b, a = iirnotch(50.0, 30.0, fs)
    sig  = filtfilt(b, a, sig)
    mx   = np.max(np.abs(sig))
    if mx > 0: sig = sig / mx
    return sig

rec_t   = wfdb.rdrecord('mitbih/100')
raw     = rec_t.p_signal[:, 0]
cleaned = preprocess(raw)

fig, ax = plt.subplots(2, 1, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')
ax[0].plot(raw[:1800],     color='#F44336', lw=0.8, label='Raw')
ax[1].plot(cleaned[:1800], color='#4CAF50', lw=0.8, label='Preprocessed')
for a in ax:
    a.set_facecolor('#161b22'); a.tick_params(colors='white')
    a.legend(labelcolor='white', facecolor='#161b22')
    for s in a.spines.values(): s.set_color('#30363d')
ax[0].set_title('Raw ECG', color='white', fontweight='bold')
ax[1].set_title('Preprocessed: Bandpass + 50Hz Notch + Normalize', color='white', fontweight='bold')
plt.tight_layout(); plt.show()
print("Cell 4 done")

## Cell 5 — Feature Extraction (17 features per beat)
> RR intervals, QRS morphology, HRV context, entropy

In [ ]:
def extract_record_features(record_id, fs=360):
    path = f'mitbih/{record_id}'
    try:
        rec = wfdb.rdrecord(path)
        ann = wfdb.rdann(path, 'atr')
    except:
        return [], []

    sig = preprocess(rec.p_signal[:, 0], fs)
    r, s = ann.sample, ann.symbol
    X, y = [], []

    for i in range(5, len(r) - 5):
        aami  = AAMI_MAP.get(s[i], None)
        if aami is None or aami == 'Q': continue
        label = LABEL_MAP.get(aami, None)
        if label is None: continue

        start = r[i] - BEFORE; end = r[i] + AFTER
        if start < 0 or end > len(sig): continue
        beat = sig[start:end]
        if len(beat) != BLEN: continue

        rr_pre  = (r[i]   - r[i-1]) / fs * 1000.0
        rr_post = (r[i+1] - r[i])   / fs * 1000.0
        if not (150 < rr_pre  < 2000): continue
        if not (150 < rr_post < 2000): continue

        mean_rr   = (rr_pre + rr_post) / 2.0
        norm_pre  = rr_pre  / mean_rr
        norm_post = rr_post / mean_rr
        rr_ratio  = rr_pre  / (rr_post + 1e-6)
        rr_diff   = (rr_post - rr_pre) / mean_rr
        ctx       = [(r[j]-r[j-1])/fs*1000 for j in range(i-5,i)
                     if 150 < (r[j]-r[j-1])/fs*1000 < 2000]
        ctx_mean  = np.mean(ctx) if ctx else mean_rr
        rr_dev    = (rr_pre - ctx_mean) / (ctx_mean + 1e-6)

        r_idx  = BEFORE
        r_amp  = beat[r_idx]
        r_abs  = max(abs(r_amp), 1e-6)
        morph  = [float(np.clip(beat[r_idx+o]/r_abs if 0<=r_idx+o<BLEN else 0,-5,5))
                  for o in [-10,-5,0,5,10]]

        qs, qe     = max(0,r_idx-20), min(BLEN,r_idx+20)
        qrs_width  = np.sum(np.abs(beat)>0.1)/fs*1000.0
        qrs_energy = float(np.sum(beat[qs:qe]**2))
        beat_std   = float(np.std(beat))
        beat_skew  = float(np.clip(skew(beat),-5,5))
        beat_kurt  = float(np.clip(kurtosis(beat),-5,20))
        beat_ent   = float(sp_entropy(np.abs(beat)+1e-10))

        feats = [
            float(np.clip(norm_pre,0,5)),   float(np.clip(norm_post,0,5)),
            float(np.clip(rr_ratio,0,5)),   float(np.clip(rr_diff,-3,3)),
            float(np.clip(rr_dev,-3,3)),
            morph[0],morph[1],morph[2],morph[3],morph[4],
            float(np.clip(qrs_width,0,300)),float(np.clip(qrs_energy,0,100)),
            float(np.clip(r_amp,-3,3)),
            float(np.clip(beat_std,0,5)),   float(np.clip(beat_skew,-5,5)),
            float(np.clip(beat_kurt,-5,20)),float(np.clip(beat_ent,0,10)),
        ]
        X.append(feats); y.append(label)
    return X, y

FEATURE_NAMES = [
    'norm_rr_pre','norm_rr_post','rr_ratio','rr_diff','rr_dev_ctx',
    'morph_m10','morph_m5','morph_0','morph_p5','morph_p10',
    'qrs_width_ms','qrs_energy','r_amp',
    'beat_std','beat_skew','beat_kurt','beat_entropy'
]
print(f"Features per beat: {len(FEATURE_NAMES)}")
for i,n in enumerate(FEATURE_NAMES):
    print(f"  {i+1:2d}. {n}")
print("Cell 5 done")

## Cell 6 — Build DS1 + DS2 Feature Datasets
> ~3-5 minutes

In [ ]:
def build_dataset(records, desc=''):
    X_all, y_all, rec_all = [], [], []
    print(f"Building {desc}...")
    for rec_id in tqdm(records, desc=desc):
        X, y = extract_record_features(rec_id)
        if len(X):
            X_all.extend(X); y_all.extend(y)
            rec_all.extend([rec_id]*len(X))
    X_arr = np.array(X_all, dtype=np.float32)
    y_arr = np.array(y_all, dtype=np.int32)
    n0,n1 = np.sum(y_arr==0), np.sum(y_arr==1)
    print(f"  Beats    : {len(X_arr):,}")
    print(f"  Normal   : {n0:,} ({n0/len(X_arr)*100:.1f}%)")
    print(f"  Abnormal : {n1:,} ({n1/len(X_arr)*100:.1f}%)")
    return X_arr, y_arr, rec_all

X_train_raw, y_train, recs_train = build_dataset(DS1, 'DS1 Training')
X_test_raw,  y_test,  recs_test  = build_dataset(DS2, 'DS2 Testing')

print(f"\nDS1: {X_train_raw.shape}")
print(f"DS2: {X_test_raw.shape}")
print(f"NaN train: {np.sum(np.isnan(X_train_raw))}")
print(f"NaN test : {np.sum(np.isnan(X_test_raw))}")
print("Cell 6 done")

## Cell 7 — Impute + SMOTE + Scale
> SMOTE on training ONLY — test set never touched

In [ ]:
imputer     = SimpleImputer(strategy='median')
X_tr_imp    = imputer.fit_transform(X_train_raw)
X_te_imp    = imputer.transform(X_test_raw)

print(f"Before SMOTE: Normal={np.sum(y_train==0):,} Abnormal={np.sum(y_train==1):,}")

k     = max(1, min(5, np.min(np.bincount(y_train))-1))
smote = SMOTE(random_state=42, k_neighbors=k)
X_tr_bal, y_tr_bal = smote.fit_resample(X_tr_imp, y_train)

print(f"After SMOTE:  Normal={np.sum(y_tr_bal==0):,} Abnormal={np.sum(y_tr_bal==1):,}")

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_tr_bal)
X_test_sc  = scaler.transform(X_te_imp)

print(f"\nTrain : {X_train_sc.shape}")
print(f"Test  : {X_test_sc.shape}")
print("Cell 7 done")

## Cell 8 — Random Forest — Patient-Level CV + Train
> GroupKFold ensures no patient appears in both train and validation

In [ ]:
record_to_id = {r:i for i,r in enumerate(DS1)}
groups       = np.array([record_to_id.get(r,0) for r in recs_train])

rf = RandomForestClassifier(
    n_estimators=400, max_depth=20, min_samples_leaf=2,
    class_weight={0:1,1:1.5}, random_state=42, n_jobs=-1)

gkf = GroupKFold(n_splits=5)
auc_scores = []
print("5-fold Patient-Level CV on DS1...\n")

for fold,(tr_idx,va_idx) in enumerate(
        gkf.split(X_tr_imp, y_train, groups)):
    X_tr=X_tr_imp[tr_idx]; y_tr=y_train[tr_idx]
    X_va=X_tr_imp[va_idx]; y_va=y_train[va_idx]
    val_pats = list(set(np.array(recs_train)[va_idx]))
    k_f  = max(1, min(5, np.min(np.bincount(y_tr))-1))
    sm_f = SMOTE(random_state=42, k_neighbors=k_f)
    Xt,yt = sm_f.fit_resample(X_tr, y_tr)
    sc_f  = StandardScaler()
    Xt    = sc_f.fit_transform(Xt)
    Xv    = sc_f.transform(X_va)
    rf.fit(Xt, yt)
    prob     = rf.predict_proba(Xv)[:,1]
    auc_fold = roc_auc_score(y_va, prob)
    auc_scores.append(auc_fold)
    print(f"Fold {fold+1} | Patients:{val_pats} | AUC:{auc_fold:.4f}")

print(f"\nCV AUC: {np.mean(auc_scores):.4f} +/- {np.std(auc_scores):.4f}")
print(f"Expected honest range: 0.85-0.95")
print("\nFinal RF training on full DS1...")
rf.fit(X_train_sc, y_tr_bal)
print("RF training complete")

## Cell 9 — RF Evaluate on DS2 + Threshold Optimisation

In [ ]:
y_rf_prob = rf.predict_proba(X_test_sc)[:,1]
fpr_rf, tpr_rf, thrs_rf = roc_curve(y_test, y_rf_prob)

best_thresh=0.5; best_sens=0.0
for i in range(len(thrs_rf)):
    sp=1.0-fpr_rf[i]; se=tpr_rf[i]
    if sp>=0.85 and se>best_sens:
        best_sens=se; best_thresh=float(thrs_rf[i])

y_rf_pred = (y_rf_prob > best_thresh).astype(int)
cm_rf     = confusion_matrix(y_test, y_rf_pred)
tn_rf,fp_rf,fn_rf,tp_rf = cm_rf.ravel()

sensitivity_rf = tp_rf/(tp_rf+fn_rf)
specificity_rf = tn_rf/(tn_rf+fp_rf)
accuracy_rf    = (tp_rf+tn_rf)/len(y_test)
f1_rf          = f1_score(y_test, y_rf_pred)
auc_rf         = roc_auc_score(y_test, y_rf_prob)
ppv_rf         = tp_rf/(tp_rf+fp_rf) if (tp_rf+fp_rf)>0 else 0
npv_rf         = tn_rf/(tn_rf+fn_rf) if (tn_rf+fn_rf)>0 else 0

print('='*52)
print('  RANDOM FOREST — DS2 AAMI RESULTS')
print('='*52)
print(f"  Accuracy    : {accuracy_rf*100:.2f}%")
print(f"  Sensitivity : {sensitivity_rf*100:.2f}%")
print(f"  Specificity : {specificity_rf*100:.2f}%")
print(f"  F1 Score    : {f1_rf*100:.2f}%")
print(f"  AUC-ROC     : {auc_rf:.4f}")
print(f"  Threshold   : {best_thresh:.4f}")
print(f"  TP:{tp_rf:,} FP:{fp_rf:,} FN:{fn_rf:,} TN:{tn_rf:,}")
print("Cell 9 done")

## Cell 10 — SHAP Explainability
> ~2-3 minutes

In [ ]:
print("Computing SHAP values (2-3 min)...")
n_shap    = min(1000, len(X_test_sc))
idx_sh    = np.random.choice(len(X_test_sc), n_shap, replace=False)
explainer = shap.TreeExplainer(rf)
shap_vals = explainer.shap_values(X_test_sc[idx_sh])
shap_ab   = shap_vals[1] if isinstance(shap_vals,list) else shap_vals
mean_shap = np.abs(shap_ab).mean(axis=0)
s_idx     = np.argsort(mean_shap)[::-1]
total     = mean_shap.sum()

print("\nTop 10 features driving arrhythmia detection:")
for rank,i in enumerate(s_idx[:10]):
    print(f"  {rank+1:2d}. {FEATURE_NAMES[i]:<20} {mean_shap[i]/total*100:5.1f}%")

fig,ax = plt.subplots(figsize=(12,5))
fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#161b22')
top10=s_idx[:10]; vals=mean_shap[top10]
names=[FEATURE_NAMES[i] for i in top10]
clrs=plt.cm.RdYlGn_r(np.linspace(0.05,0.95,10))
ax.bar(range(10),vals,color=clrs,width=0.65)
ax.set_xticks(range(10))
ax.set_xticklabels(names,rotation=20,ha='right',color='white',fontsize=9)
ax.set_title('SHAP Feature Importance',color='white',fontweight='bold')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_visible(False)
plt.tight_layout()
plt.savefig('shap_importance.png',dpi=120,facecolor='#0d1117',bbox_inches='tight')
plt.show()
print("Cell 10 done")

## Cell 11 — Extract Raw Beats for CNN
> ~3-5 minutes

In [ ]:
def get_raw_beats(records, desc=''):
    beats_all, labels_all = [], []
    for rec_id in tqdm(records, desc=desc):
        try:
            path = f'mitbih/{rec_id}'
            rec  = wfdb.rdrecord(path)
            ann  = wfdb.rdann(path, 'atr')
            sig  = preprocess(rec.p_signal[:,0])
            r,s  = ann.sample, ann.symbol
            for i in range(5, len(r)-5):
                aami  = AAMI_MAP.get(s[i],None)
                if aami is None or aami=='Q': continue
                label = LABEL_MAP.get(aami,None)
                if label is None: continue
                start=r[i]-BEFORE; end=r[i]+AFTER
                if start<0 or end>len(sig): continue
                beat=sig[start:end]
                if len(beat)!=BLEN: continue
                mu=np.mean(beat); std=np.std(beat)
                if std>1e-6: beat=(beat-mu)/std
                beats_all.append(beat.astype(np.float32))
                labels_all.append(label)
        except: pass
    X=np.array(beats_all,dtype=np.float32)
    y=np.array(labels_all,dtype=np.int32)
    print(f"  {desc}: {len(X):,} beats | Normal:{np.sum(y==0):,} Abnormal:{np.sum(y==1):,}")
    return X,y

X_cnn_tr_raw,y_cnn_tr = get_raw_beats(DS1,'DS1')
X_cnn_te_raw,y_cnn_te = get_raw_beats(DS2,'DS2')

print("\nSMOTE balancing training set...")
X_flat  = X_cnn_tr_raw.reshape(len(X_cnn_tr_raw),-1)
k2      = max(1,min(5,np.min(np.bincount(y_cnn_tr))-1))
sm2     = SMOTE(random_state=42,k_neighbors=k2)
Xf_bal,y_cnn_bal = sm2.fit_resample(X_flat,y_cnn_tr)

X_cnn_train = Xf_bal.reshape(-1,BLEN,1).astype(np.float32)
y_cnn_train = y_cnn_bal.astype(np.float32)
X_cnn_test  = X_cnn_te_raw.reshape(-1,BLEN,1).astype(np.float32)
y_cnn_test  = y_cnn_te.astype(np.float32)

print(f"CNN Train : {X_cnn_train.shape}")
print(f"CNN Test  : {X_cnn_test.shape}")
print("Cell 11 done")

## Cell 12 — Build + Train 1D CNN
> ~5-15 minutes on T4 GPU | Architecture fits <150KB after INT8

In [ ]:
def build_cnn(input_len=BLEN):
    inp = keras.Input(shape=(input_len,1), name='ecg_input')
    x = layers.Conv1D(32,5,padding='same',use_bias=False,name='conv1')(inp)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Activation('relu',name='relu1')(x)
    x = layers.MaxPooling1D(2,name='pool1')(x)
    x = layers.Dropout(0.2,name='drop1')(x)
    x = layers.Conv1D(64,5,padding='same',use_bias=False,name='conv2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Activation('relu',name='relu2')(x)
    x = layers.MaxPooling1D(2,name='pool2')(x)
    x = layers.Dropout(0.2,name='drop2')(x)
    x = layers.Conv1D(32,3,padding='same',use_bias=False,name='conv3')(x)
    x = layers.BatchNormalization(name='bn3')(x)
    x = layers.Activation('relu',name='relu3')(x)
    x = layers.GlobalAveragePooling1D(name='gap')(x)
    x = layers.Dropout(0.3,name='drop3')(x)
    x = layers.Dense(32,activation='relu',name='dense1')(x)
    x = layers.Dropout(0.2,name='drop4')(x)
    out = layers.Dense(1,activation='sigmoid',name='output')(x)
    return keras.Model(inp,out,name='ArrhythmiaGuard_CNN')

cnn = build_cnn()
cnn.summary()
print(f"\nParameters   : {cnn.count_params():,}")
print(f"Est. INT8 KB : ~{cnn.count_params()/1024:.0f}KB")

cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.AUC(name='auc'),
             keras.metrics.Recall(name='sensitivity')])

os.makedirs('saved_models', exist_ok=True)
callbacks_cnn = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=7,
        restore_best_weights=True, mode='max', verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(
        'saved_models/best_cnn.keras',
        monitor='val_auc', save_best_only=True, mode='max', verbose=0)
]

print("\nTraining 1D CNN...")
history_cnn = cnn.fit(
    X_cnn_train, y_cnn_train,
    epochs=50, batch_size=256,
    validation_split=0.15,
    callbacks=callbacks_cnn, verbose=1)

fig,ax = plt.subplots(1,2,figsize=(14,4))
fig.patch.set_facecolor('#0d1117')
for a in ax:
    a.set_facecolor('#161b22'); a.tick_params(colors='white')
    for s in a.spines.values(): s.set_color('#30363d')
ax[0].plot(history_cnn.history['loss'],color='#F44336',label='Train')
ax[0].plot(history_cnn.history['val_loss'],color='#4CAF50',label='Val')
ax[0].set_title('Loss',color='white',fontweight='bold')
ax[0].legend(labelcolor='white',facecolor='#161b22')
ax[1].plot(history_cnn.history['auc'],color='#2196F3',label='Train')
ax[1].plot(history_cnn.history['val_auc'],color='#FF9800',label='Val')
ax[1].set_title('AUC',color='white',fontweight='bold')
ax[1].legend(labelcolor='white',facecolor='#161b22')
plt.suptitle('CNN Training Curves',color='white',fontweight='bold')
plt.tight_layout()
plt.savefig('cnn_training_curves.png',dpi=120,facecolor='#0d1117',bbox_inches='tight')
plt.show()
print("Cell 12 done")

## Cell 13 — Evaluate CNN on DS2

In [ ]:
print("Evaluating CNN on DS2...")
y_cnn_prob = cnn.predict(X_cnn_test,batch_size=256,verbose=0).flatten()
fpr_cn,tpr_cn,thrs_cn = roc_curve(y_cnn_test.astype(int), y_cnn_prob)

cnn_best_t=0.5; cnn_best_s=0.0
for i in range(len(thrs_cn)):
    sp=1.0-fpr_cn[i]; se=tpr_cn[i]
    if sp>=0.85 and se>cnn_best_s:
        cnn_best_s=se; cnn_best_t=float(thrs_cn[i])

y_cnn_pred=(y_cnn_prob>cnn_best_t).astype(int)
cm_cnn=confusion_matrix(y_cnn_test.astype(int),y_cnn_pred)
tn_c,fp_c,fn_c,tp_c=cm_cnn.ravel()

cnn_sens = tp_c/(tp_c+fn_c)
cnn_spec = tn_c/(tn_c+fp_c)
cnn_acc  = (tp_c+tn_c)/len(y_cnn_test)
cnn_f1   = f1_score(y_cnn_test.astype(int),y_cnn_pred)
cnn_auc  = roc_auc_score(y_cnn_test.astype(int),y_cnn_prob)
cnn_ppv  = tp_c/(tp_c+fp_c) if (tp_c+fp_c)>0 else 0
cnn_npv  = tn_c/(tn_c+fn_c) if (tn_c+fn_c)>0 else 0

print('='*52)
print('  1D CNN — DS2 AAMI RESULTS')
print('='*52)
print(f"  Accuracy    : {cnn_acc*100:.2f}%")
print(f"  Sensitivity : {cnn_sens*100:.2f}%")
print(f"  Specificity : {cnn_spec*100:.2f}%")
print(f"  F1 Score    : {cnn_f1*100:.2f}%")
print(f"  AUC-ROC     : {cnn_auc:.4f}")
print(f"  Threshold   : {cnn_best_t:.4f}")
print(f"  TP:{tp_c:,} FP:{fp_c:,} FN:{fn_c:,} TN:{tn_c:,}")
print("\nAll variables defined for Cell 14")
print("Cell 13 done")

## Cell 14 — Full Results Visualisation

In [ ]:
fig = plt.figure(figsize=(20,14))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(3,3,figure=fig,hspace=0.45,wspace=0.35)
fig.suptitle('ArrhythmiaGuard — RF + 1D CNN | AAMI EC57 Inter-Patient',
             fontsize=14,fontweight='bold',color='white',y=0.98)

# RF confusion matrix
ax1=fig.add_subplot(gs[0,0]); ax1.set_facecolor('#161b22')
cmd=np.array([[tn_rf,fp_rf],[fn_rf,tp_rf]])
ax1.imshow(cmd,cmap='Blues',interpolation='nearest')
ax1.set_title('RF Confusion Matrix',color='white',fontweight='bold',fontsize=10)
ax1.set_xticks([0,1]); ax1.set_yticks([0,1])
ax1.set_xticklabels(['Pred N','Pred A'],color='white',fontsize=9)
ax1.set_yticklabels(['Act N','Act A'],color='white',fontsize=9)
for i in range(2):
    for j in range(2):
        v=cmd[i,j]
        ax1.text(j,i,f'{v:,}',ha='center',va='center',
                 fontsize=11,fontweight='bold',
                 color='white' if v>max(tn_rf,tp_rf)//2 else 'black')

# CNN confusion matrix
ax2=fig.add_subplot(gs[0,1]); ax2.set_facecolor('#161b22')
cmc=np.array([[tn_c,fp_c],[fn_c,tp_c]])
ax2.imshow(cmc,cmap='Greens',interpolation='nearest')
ax2.set_title('CNN Confusion Matrix',color='white',fontweight='bold',fontsize=10)
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(['Pred N','Pred A'],color='white',fontsize=9)
ax2.set_yticklabels(['Act N','Act A'],color='white',fontsize=9)
for i in range(2):
    for j in range(2):
        v=cmc[i,j]
        ax2.text(j,i,f'{v:,}',ha='center',va='center',
                 fontsize=11,fontweight='bold',
                 color='white' if v>max(tn_c,tp_c)//2 else 'black')

# ROC
ax3=fig.add_subplot(gs[0,2]); ax3.set_facecolor('#161b22')
ax3.plot(fpr_rf,tpr_rf,color='#2196F3',lw=2,label=f'RF  AUC={auc_rf:.3f}')
ax3.plot(fpr_cn,tpr_cn,color='#4CAF50',lw=2,label=f'CNN AUC={cnn_auc:.3f}')
ax3.plot([0,1],[0,1],'w--',lw=1,alpha=0.4)
ax3.fill_between(fpr_cn,tpr_cn,alpha=0.08,color='#4CAF50')
ax3.set_title('ROC Curves',color='white',fontweight='bold',fontsize=10)
ax3.set_xlabel('FPR',color='white',fontsize=9)
ax3.set_ylabel('TPR',color='white',fontsize=9)
ax3.legend(facecolor='#161b22',labelcolor='white',fontsize=9)
ax3.tick_params(colors='white')
for s in ax3.spines.values(): s.set_color('#30363d')

# Metrics table
ax4=fig.add_subplot(gs[1,0]); ax4.set_facecolor('#161b22'); ax4.axis('off')
ax4.set_title('RF vs CNN Metrics',color='white',fontweight='bold',fontsize=10)
mets=[('Accuracy',f'{accuracy_rf*100:.1f}%',f'{cnn_acc*100:.1f}%'),
      ('Sensitivity',f'{sensitivity_rf*100:.1f}%',f'{cnn_sens*100:.1f}%'),
      ('Specificity',f'{specificity_rf*100:.1f}%',f'{cnn_spec*100:.1f}%'),
      ('F1 Score',f'{f1_rf*100:.1f}%',f'{cnn_f1*100:.1f}%'),
      ('AUC-ROC',f'{auc_rf:.3f}',f'{cnn_auc:.3f}')]
ax4.set_xlim(0,1); ax4.set_ylim(0,7)
ax4.text(0.35,6.5,'RF',color='#2196F3',fontweight='bold',ha='center',fontsize=10)
ax4.text(0.75,6.5,'CNN',color='#4CAF50',fontweight='bold',ha='center',fontsize=10)
for i,(nm,rv,cv) in enumerate(mets):
    yp=5.5-i
    ax4.text(0.02,yp,nm,color='#8b949e',fontsize=10)
    ax4.text(0.35,yp,rv,color='#2196F3',fontsize=11,fontweight='bold',ha='center')
    ax4.text(0.75,yp,cv,color='#4CAF50',fontsize=11,fontweight='bold',ha='center')

# SHAP
ax5=fig.add_subplot(gs[1,1:]); ax5.set_facecolor('#161b22')
top8=s_idx[:8]; vals8=mean_shap[top8]; nm8=[FEATURE_NAMES[i] for i in top8]
tot8=vals8.sum(); clr8=plt.cm.RdYlGn_r(np.linspace(0.05,0.95,8))
bars8=ax5.bar(range(8),vals8,color=clr8,width=0.65)
ax5.set_xticks(range(8))
ax5.set_xticklabels(nm8,rotation=15,ha='right',color='white',fontsize=9)
ax5.set_title('SHAP Feature Importance',color='white',fontweight='bold',fontsize=10)
ax5.tick_params(colors='white')
for sp in ax5.spines.values(): sp.set_visible(False)
for bar,val in zip(bars8,vals8):
    ax5.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.0001,
             f'{val/tot8*100:.1f}%',ha='center',va='bottom',color='white',fontsize=8)

# CNN training
ax6=fig.add_subplot(gs[2,0]); ax6.set_facecolor('#161b22')
ax6.plot(history_cnn.history['val_auc'],color='#4CAF50',lw=2,label='Val AUC')
ax6.plot(history_cnn.history['auc'],color='#4CAF50',lw=1,ls='--',alpha=0.5,label='Train AUC')
ax6.set_title('CNN AUC Curve',color='white',fontweight='bold',fontsize=10)
ax6.legend(facecolor='#161b22',labelcolor='white',fontsize=8)
ax6.tick_params(colors='white')
for s in ax6.spines.values(): s.set_color('#30363d')

# Cost
ax7=fig.add_subplot(gs[2,1]); ax7.set_facecolor('#161b22')
devs=['Holter\nMonitor','Apple\nWatch','KardiaMobile','ArrhythmiaGuard']
costs=[300000,75000,20000,5000]; clrz=['#F44336','#FF9800','#FFC107','#4CAF50']
b3=ax7.bar(range(4),costs,color=clrz,edgecolor='none',width=0.55)
ax7.set_xticks(range(4)); ax7.set_xticklabels(devs,color='white',fontsize=9)
ax7.set_title('Cost Comparison (INR)',color='white',fontweight='bold',fontsize=10)
ax7.tick_params(colors='white')
for s in ax7.spines.values(): s.set_visible(False)
for bar,cost in zip(b3,costs):
    ax7.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1500,
             f'Rs.{cost:,}',ha='center',va='bottom',color='white',fontsize=9,fontweight='bold')

# Architecture
ax8=fig.add_subplot(gs[2,2]); ax8.set_facecolor('#161b22'); ax8.axis('off')
ax8.set_title('System Architecture',color='white',fontweight='bold',fontsize=10)
arch=[('AD8232 ECG Sensor','#FF9800'),('ESP32-S3 N16R8','#2196F3'),
      ('TFLite 1D CNN','#4CAF50'),('SSD1306 OLED','#9C27B0'),
      ('LED Indicators','#F44336'),('Streamlit Dashboard','#00BCD4'),
      ('RF + SHAP Explain','#FFD700')]
ax8.set_xlim(0,1); ax8.set_ylim(0,len(arch)+1)
for i,(name,clr) in enumerate(arch):
    yp=len(arch)-i
    ax8.barh(yp,0.8,left=0.1,height=0.6,color=clr,alpha=0.8)
    ax8.text(0.5,yp,name,ha='center',va='center',color='white',fontsize=9,fontweight='bold')

plt.savefig('ArrhythmiaGuard_Results.png',dpi=150,bbox_inches='tight',facecolor='#0d1117')
plt.show()

print('\nPOSTER NUMBERS:')
print(f"RF  Acc:{accuracy_rf*100:.1f}% Sens:{sensitivity_rf*100:.1f}% AUC:{auc_rf:.3f}")
print(f"CNN Acc:{cnn_acc*100:.1f}% Sens:{cnn_sens*100:.1f}% AUC:{cnn_auc:.3f}")
print("Cell 14 done")

## Cell 15 — TFLite INT8 → C Header for ESP32-S3
> INT8 weights + float32 I/O avoids known zero-output bug on ESP32-S3

In [ ]:
os.makedirs('arduino_export', exist_ok=True)
print("Converting to TFLite INT8 weights, float32 I/O...")

def representative_dataset():
    idx = np.random.choice(len(X_cnn_train),
                            min(500,len(X_cnn_train)),replace=False)
    for i in idx:
        yield [X_cnn_train[i:i+1]]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
# NOTE: NOT setting inference_input/output_type to int8
# Keeps I/O as float32 — prevents zero-output bug on ESP32-S3
tflite_model = converter.convert()

with open('saved_models/cnn_model.tflite','wb') as f:
    f.write(tflite_model)
model_kb = len(tflite_model)/1024
print(f"TFLite model : {model_kb:.1f}KB  (16384KB available on N16R8)")

# Verify on PC
print("\nVerifying on PC...")
interp=tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
inp_d=interp.get_input_details(); out_d=interp.get_output_details()
print(f"Input  : {inp_d[0]['shape']} dtype={inp_d[0]['dtype']}")
print(f"Output : {out_d[0]['shape']} dtype={out_d[0]['dtype']}")
correct=0
for i in range(20):
    s=X_cnn_test[i:i+1]
    interp.set_tensor(inp_d[0]['index'],s)
    interp.invoke()
    prob=interp.get_tensor(out_d[0]['index'])[0][0]
    pred=1 if prob>cnn_best_t else 0
    true=int(y_cnn_test[i])
    if pred==true: correct+=1
print(f"PC verify : {correct}/20 correct")
if correct < 14:
    print("WARNING: Consider retraining CNN")
else:
    print("Verification passed")

# Generate C header
print("\nGenerating cnn_model.h...")
lines=[
    '// cnn_model.h — Auto-generated by ArrhythmiaGuard Colab',
    '// DO NOT EDIT MANUALLY',
    f'// CNN Accuracy: {cnn_acc*100:.1f}%  Sensitivity: {cnn_sens*100:.1f}%  AUC: {cnn_auc:.4f}',
    f'// Model size: {model_kb:.1f}KB  Threshold: {cnn_best_t:.6f}',
    f'// Input/Output: float32  Quantization: INT8 weights',
    f'// Target: ESP32-S3 N16R8',
    '','#ifndef CNN_MODEL_H','#define CNN_MODEL_H','','#include <stdint.h>','',
    '// Inference parameters',
    f'#define CNN_THRESHOLD      {cnn_best_t:.6f}f',
    f'#define CNN_BEAT_LEN       {BLEN}',
    f'#define CNN_BEAT_BEFORE    {BEFORE}',
    f'#define CNN_SAMPLE_RATE    360',
    f'#define CNN_SMOOTH_WINDOW  5',
    f'#define CNN_ARENA_SIZE     (200 * 1024)',
    '',
    f'const unsigned int cnn_model_len = {len(tflite_model)};',
    '','alignas(8) const unsigned char cnn_model_data[] = {',
]
for i in range(0,len(tflite_model),12):
    chunk=tflite_model[i:i+12]
    hex_str=', '.join([f'0x{b:02x}' for b in chunk])
    comma=',' if i+12<len(tflite_model) else ''
    lines.append(f'  {hex_str}{comma}')
lines+=['};','','#endif // CNN_MODEL_H']
with open('arduino_export/cnn_model.h','w') as f:
    f.write('\n'.join(lines))
print(f"cnn_model.h : {os.path.getsize('arduino_export/cnn_model.h')/1024:.0f}KB")
print("Cell 15 done")

## Cell 16 — Save All Models + Metrics

In [ ]:
joblib.dump(rf,      'saved_models/rf_model.pkl')
joblib.dump(scaler,  'saved_models/scaler.pkl')
joblib.dump(imputer, 'saved_models/imputer.pkl')
cnn.save('saved_models/cnn_full_model.keras')

all_metrics={
    'random_forest':{'accuracy':float(accuracy_rf),'sensitivity':float(sensitivity_rf),
                     'specificity':float(specificity_rf),'f1':float(f1_rf),
                     'auc':float(auc_rf),'threshold':float(best_thresh),
                     'role':'explainability_dashboard'},
    'cnn_1d':{'accuracy':float(cnn_acc),'sensitivity':float(cnn_sens),
              'specificity':float(cnn_spec),'f1':float(cnn_f1),
              'auc':float(cnn_auc),'threshold':float(cnn_best_t),
              'model_kb':float(model_kb),'role':'edge_deployment_esp32s3'},
    'deployment':{'board':'ESP32-S3 N16R8','framework':'TFLite Micro',
                  'quantization':'INT8_weights_float32_IO',
                  'arena_kb':200,'beat_len':BLEN,'sample_rate':FS},
    'validation':{'protocol':'AAMI EC57','split':'De Chazal DS1/DS2',
                  'train_pats':len(DS1),'test_pats':len(DS2),'smote':True},
    'features':FEATURE_NAMES
}
with open('saved_models/all_metrics.json','w') as f:
    json.dump(all_metrics,f,indent=2)

print("Saved:")
print("  rf_model.pkl  scaler.pkl  imputer.pkl")
print("  cnn_full_model.keras  cnn_model.tflite")
print("  all_metrics.json  arduino_export/cnn_model.h")
print("Cell 16 done")

## Cell 17 — Download Everything

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('ArrhythmiaGuard_Arduino','zip','arduino_export')
shutil.make_archive('ArrhythmiaGuard_Models','zip','saved_models')

print("Downloading...")
files.download('arduino_export/cnn_model.h')
files.download('ArrhythmiaGuard_Arduino.zip')
files.download('ArrhythmiaGuard_Models.zip')
files.download('saved_models/all_metrics.json')
files.download('ArrhythmiaGuard_Results.png')
files.download('cnn_training_curves.png')
files.download('shap_importance.png')

print("\nAll downloads complete")
print('='*52)
print('FINAL POSTER NUMBERS')
print('='*52)
print(f"Random Forest (Explainability+Dashboard):")
print(f"  Accuracy    : {accuracy_rf*100:.1f}%")
print(f"  Sensitivity : {sensitivity_rf*100:.1f}%")
print(f"  Specificity : {specificity_rf*100:.1f}%")
print(f"  AUC-ROC     : {auc_rf:.3f}")
print(f"\n1D CNN (ESP32-S3 N16R8 Edge Deployment):")
print(f"  Accuracy    : {cnn_acc*100:.1f}%")
print(f"  Sensitivity : {cnn_sens*100:.1f}%")
print(f"  Specificity : {cnn_spec*100:.1f}%")
print(f"  AUC-ROC     : {cnn_auc:.3f}")
print(f"  Model size  : {model_kb:.0f}KB INT8")
print(f"  Threshold   : {cnn_best_t:.4f}")
print('='*52)
print("Validation : AAMI EC57 Inter-Patient")
print("Protocol   : De Chazal DS1/DS2")
print("Board      : ESP32-S3 N16R8")